# Chuyên đề 03 · Kiểm chứng nghiên cứu mới về mô hình sinh và hậu huấn luyện: sao chép, sụp đổ, quên, chuỗi suy nghĩ

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd03-nghien-cuu-sinh-va-hau-huan-luyen.ipynb)

Notebook đi kèm [Chuyên đề 03 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/nghien-cuu-sinh-va-hau-huan-luyen/). Năm kết quả nghiên cứu gần đây về mô hình sinh và
về việc huấn luyện tiếp một mô hình đã có, mỗi kết quả kiểm lại trên một bài toán nhỏ đủ để chạy trên CPU. Như
Chuyên đề 02, ta đặt lại thí nghiệm, chạy nhiều seed, và ghi rõ kết quả **khớp**, **không khớp** hay **tùy điều
kiện**.

**Bạn sẽ làm:**
1. Dựng một "mô hình khuếch tán hoàn hảo" bằng công thức, thấy nó chỉ biết sao chép, rồi thêm tính cục bộ và thấy nó
   tạo ra chữ số mới (Kamb & Ganguli, 2025).
2. Đo thời điểm một mô hình khuếch tán thật bắt đầu sao chép, theo số ảnh huấn luyện (Bonnaire và cộng sự, 2025).
3. Huấn luyện nhiều thế hệ mô hình trên dữ liệu do thế hệ trước sinh ra: thay thế dữ liệu thì sụp đổ, giữ lại thì không
   (Shumailov và cộng sự, 2024; Gerstgrasser và cộng sự, 2024).
4. So sánh SFT và học tăng cường khi huấn luyện tiếp một mô hình: cái nào quên nhiều hơn, và vì sao (Shenfeld và cộng
   sự, 2025).
5. Dạy một transformer nhỏ tính tính chẵn lẻ của vài bit bí mật, có và không có chuỗi suy nghĩ (Kim & Suzuki, 2025).

**Cần biết trước:** [Bài 21 · Khuếch tán & flow matching](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/diffusion/),
[Bài 22 · Tiền huấn luyện & tinh chỉnh](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/tien-huan-luyen/),
[Bài 26 · Từ mô hình ngôn ngữ đến trợ lý](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/hau-huan-luyen/) và
[Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/).
**Thời gian chạy:** khoảng 1 phút trên CPU, kể cả tải dữ liệu lần đầu.

Notebook dùng **PyTorch**. Mỗi mục chạy trực tiếp một phiên bản ngắn, rồi nạp kết quả đầy đủ (nhiều seed, chạy lâu
hơn) mà các script `reel/research/*.py` của khóa học đã tính sẵn, cùng mã và cùng seed.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

In [ ]:
import gzip
import hashlib
import json
import math
import os
import re
import urllib.request
from itertools import pairwise
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn

torch.set_num_threads(2)

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # âm → 0 → dương
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=1):
    return vn(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def vn_tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Nhãn trục kiểu Việt Nam; gọi ngay trước plt.show(). Trục log ngắn lấy vạch 1–2–5, trục log dài lấy lũy thừa của 10."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))

## 2 · Dữ liệu

MNIST và Fashion-MNIST (đầy đủ bốn tệp mỗi bộ, có kiểm MD5), *Truyện Kiều* từ Wikisource (ghim ở phiên bản 205132,
như Bài 15 và 17), và sáu tệp kết quả của các script.

In [ ]:
MNIST_URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
MNIST_FILES = {
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
FASHION_URL = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"
FASHION_FILES = {
    "train-images-idx3-ubyte.gz": "8d4fb7e6c68d591d4c3dfef9ec88bf0d",
    "train-labels-idx1-ubyte.gz": "25c81989df183df01b3e8a0aad5dffbe",
    "t10k-images-idx3-ubyte.gz": "bef4ecab320f06d8554ea6380940ec79",
    "t10k-labels-idx1-ubyte.gz": "bb300cfdad3c16e7a12a480ee83cd310",
}
KIEU_SRC = "https://vi.wikisource.org/w/index.php?oldid=205132&action=raw"
KIEU_RAW = Path("truyen_kieu_205132.wiki")
opener = urllib.request.build_opener()
opener.addheaders = [("User-Agent", "HocSauReels-notebook/1.0 (education; https://d3qr1cjln6qqtk.cloudfront.net/)")]
urllib.request.install_opener(opener)  # Wikimedia từ chối các yêu cầu không có User-Agent
for folder, url, files in (("mnist", MNIST_URL, MNIST_FILES), ("fashion", FASHION_URL, FASHION_FILES)):
    Path(folder).mkdir(exist_ok=True)
    for name, md5 in files.items():
        p = Path(folder) / name
        if not p.exists():
            urllib.request.urlretrieve(url + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{p} bị hỏng, hãy xóa và tải lại"
if not KIEU_RAW.exists():
    urllib.request.urlretrieve(KIEU_SRC, KIEU_RAW)
assert hashlib.md5(KIEU_RAW.read_bytes()).hexdigest() == "d4090daac094841888ee2b2b18661065"


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


LOCAL = fetch_json("research-diffusion-local.json", "64522a8076946d83c1d1b3ca78dc7ce1")
MEMO = fetch_json("research-memo.json", "948f96f67edb5ffd8534b9a41a9ebed6")
COLLAPSE = fetch_json("research-collapse.json", "0202c9477632ae5b0692ca67ff8225fe")
RAZOR = fetch_json("research-razor.json", "f5945b63dda5864e63e6d44b9c868479")
COT = fetch_json("research-cot.json", "42729fb56fdd9d6c96e7304f4a016956")

In [ ]:
def idx(folder, kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(Path(folder) / f"{kind}-images-idx3-ubyte.gz") as f:
        x = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(Path(folder) / f"{kind}-labels-idx1-ubyte.gz") as f:
        y = np.frombuffer(f.read(), np.uint8, offset=8)
    return x, y.astype(np.int64)


mnist_tr, mnist_ytr = idx("mnist", "train")
mnist_te, mnist_yte = idx("mnist", "t10k")
fash_tr, fash_ytr = idx("fashion", "train")
fash_te, fash_yte = idx("fashion", "t10k")
print("MNIST", mnist_tr.shape, mnist_te.shape, "· Fashion-MNIST", fash_tr.shape, fash_te.shape)

## 3 · Mô hình khuếch tán: sao chép hay sáng tạo? Một lời giải bằng công thức

**Nhận định** (Kamb & Ganguli, ICML 2025). Bài 21 đã gặp **bộ khử nhiễu chính xác**: lời giải đúng của bài toán khử
nhiễu trên một tập huấn luyện hữu hạn. Nó là trung bình có trọng số softmax của các ảnh huấn luyện, và lấy mẫu bằng nó
chỉ có thể trả về đúng một ảnh huấn luyện. Một mô hình khuếch tán học *hoàn hảo* mục tiêu của nó sẽ chỉ sao chép. Vậy
vì sao mô hình thật lại tạo ra ảnh mới? Kamb & Ganguli chỉ ra rằng chỉ cần thêm hai ràng buộc mà mạng tích chập có
sẵn: **tính cục bộ** (mỗi điểm ảnh đầu ra chỉ nhìn một mảng $P \times P$ quanh nó) và, nếu muốn, **bất biến tịnh
tiến**. Khi đó lời giải vẫn viết được bằng công thức, và nó tạo ra ảnh mới như những bức **khảm** ghép từ các mảng của
ảnh huấn luyện. Họ còn thấy các "máy" công thức này dự đoán khá sát đầu ra của mạng U-Net đã huấn luyện. Một bài sau
(Lukoianov và cộng sự, 2025) phản biện rằng tính cục bộ không cần đến mạng: nó đã có sẵn trong tương quan giữa các
điểm ảnh của dữ liệu.

Thí nghiệm: 1.000 ảnh MNIST huấn luyện, điểm ảnh đưa về $[-1, 1]$; đường nhiễu như Bài 21,
$x_t = (1 - t)\,x_0 + t\,\varepsilon$, với $t = 1$ là nhiễu thuần. Không huấn luyện gì cả:
- **lý tưởng:** $\hat x_0(x_t) = \sum_i w_i x_i$ với $w_i \propto \exp\!\big(-\lVert x_t - (1-t)x_i\rVert^2 / 2t^2\big)$
  trên **cả ảnh**;
- **cục bộ:** cùng softmax đó, nhưng tính riêng cho từng điểm ảnh trên mảng $P \times P$ quanh nó, nên mỗi điểm ảnh
  có thể "nghe theo" một ảnh huấn luyện khác.

Lấy mẫu bằng 20 bước tất định từ $t = 1$ về 0, như DDIM. Một mẫu được coi là **bản sao** nếu nó gần ảnh huấn luyện
gần nhất hơn 3 lần so với ảnh gần thứ hai (quy tắc của Bài 21).

In [ ]:
TRAIN = mnist_tr[:1000].astype(np.float64) / 127.5 - 1
TIMES = np.linspace(1, 0, 21) ** 1.5  # 20 bước, dày hơn gần t = 0, nơi các chi tiết được quyết định


def softmax_w(D, t):
    """Softmax theo trục 0 (các ảnh huấn luyện) của −D / 2t²; tổng các trọng số bằng 1 ở mọi điểm ảnh."""
    L = -D / (2 * t * t)
    w = np.exp(L - L.max(0, keepdims=True))
    return w / w.sum(0, keepdims=True)


def ideal(xt, t):
    D = ((xt[None] - (1 - t) * TRAIN) ** 2).sum((1, 2))  # một khoảng cách cho mỗi ảnh huấn luyện
    return np.tensordot(softmax_w(D, t), TRAIN, 1)


def boxsum(a, P):
    """Tổng của a trên cửa sổ P×P quanh mỗi điểm ảnh (cắt ở mép), bằng tổng tích lũy hai chiều. a: (n, 28, 28)."""
    r = P // 2
    c = np.pad(a, ((0, 0), (1, 0), (1, 0))).cumsum(1).cumsum(2)
    lo, hi = np.clip(np.arange(28) - r, 0, 28), np.clip(np.arange(28) + r + 1, 0, 28)
    return c[:, hi][:, :, hi] - c[:, lo][:, :, hi] - c[:, hi][:, :, lo] + c[:, lo][:, :, lo]


def local(xt, t, P):
    w = softmax_w(boxsum((xt[None] - (1 - t) * TRAIN) ** 2, P), t)  # một khoảng cách cho mỗi (ảnh, điểm ảnh)
    return (w * TRAIN).sum(0)


def sample(den, seed):
    """Từ nhiễu thuần về ảnh: ở mỗi bước, đoán x̂₀, suy ra ε, rồi đặt x về thời điểm kế tiếp."""
    x = np.random.default_rng(seed).standard_normal((28, 28))
    for t, s in pairwise(TIMES):
        x0 = den(x, t)
        eps = (x - (1 - t) * x0) / t
        x = (1 - s) * x0 + s * eps
    return x


def nearest(x):
    """Khoảng cách tới ảnh huấn luyện gần nhất, tỉ số gần nhất / gần thứ hai, và chỉ số của ảnh gần nhất."""
    d = np.sqrt(((TRAIN - x) ** 2).sum((1, 2)))
    i = np.argsort(d)
    return d[i[0]], d[i[0]] / d[i[1]], int(i[0])

### 🤔 Dự đoán trước
Lấy mẫu từ bộ khử nhiễu lý tưởng và từ bộ khử nhiễu cục bộ với $P = 9$, cùng một nhiễu ban đầu. Mỗi cách cho ra bao
nhiêu phần trăm bản sao?

In [ ]:
live = {"lý tưởng": [sample(ideal, s) for s in (0, 1)],
        "cục bộ, P = 9": [sample(lambda z, t: local(z, t, 9), s) for s in (0, 1)]}  # fmt: skip
fig, axes = plt.subplots(2, 4, figsize=(7, 3.8))
for row, (name, xs) in enumerate(live.items()):
    for j, x in enumerate(xs):
        d, ratio, i = nearest(x)
        axes[row, 2 * j].imshow(x, cmap="gray", vmin=-1, vmax=1)
        axes[row, 2 * j].set_title(f"{name}\nmẫu {j + 1}", fontsize=8)
        axes[row, 2 * j + 1].imshow(TRAIN[i], cmap="gray", vmin=-1, vmax=1)
        axes[row, 2 * j + 1].set_title(f"ảnh huấn luyện gần nhất\nkhoảng cách {vn(d, 1)}", fontsize=8)
        print(f"{name:14} mẫu {j + 1}: gần nhất {vn(d, 2)}, tỉ số {vn(ratio, 3)} → "
              f"{'BẢN SAO' if ratio < 1 / 3 else 'ảnh mới'}")  # fmt: skip
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()
for j in range(2):
    assert nearest(live["lý tưởng"][j])[2] == LOCAL["ideal"]["nnIndex"][j], "khác với kết quả đã lưu"
    assert abs(nearest(live["cục bộ, P = 9"][j])[0] / LOCAL["local"]["9"]["nn"][j] - 1) < 0.02, (
        "khác với kết quả đã lưu"
    )

Hàng trên: bộ khử nhiễu lý tưởng trả về **đúng** một ảnh huấn luyện (khoảng cách 0 tới ảnh bên cạnh nó). Hàng dưới:
cùng nhiễu ban đầu, nhưng mỗi điểm ảnh chỉ nhìn mảng 9 × 9 quanh nó; kết quả là một nét chữ mới, không trùng ảnh huấn
luyện nào. Kết quả đầy đủ, 16 mẫu cho mỗi cỡ mảng:

In [ ]:
print(f"{'':20} bản sao   khoảng cách tới ảnh huấn luyện gần nhất (trung vị)")
print(f"{'ảnh kiểm tra thật':20} {pct(LOCAL['test']['copy'], 0):>6}   {vn(np.median(LOCAL['test']['nn']), 2)}")
print(f"{'lý tưởng':20} {pct(LOCAL['ideal']['copy'], 0):>6}   {vn(np.median(LOCAL['ideal']['nn']), 2)}")
for P in LOCAL["patches"]:
    r_ = LOCAL["local"][str(P)]
    print(f"{f'cục bộ, P = {P}':20} {pct(r_['copy'], 0):>6}   {vn(np.median(r_['nn']), 2)}")
print(f"{'cục bộ + tịnh tiến, P = 5 (256 ảnh)':20} {pct(LOCAL['els5']['copy'], 0):>6}   "
      f"{vn(np.median(LOCAL['els5']['nn']), 2)}")  # fmt: skip

rows = [("lý tưởng", LOCAL["ideal"]["img"])] + [(f"P = {P}", LOCAL["local"][str(P)]["img"]) for P in (3, 5, 9, 13)]
rows.append(("tịnh tiến, P = 5", LOCAL["els5"]["img"]))
fig, axes = plt.subplots(len(rows), 8, figsize=(8, 1.1 * len(rows)))
for i, (name, imgs) in enumerate(rows):
    for j in range(8):
        axes[i, j].imshow(np.array(imgs[j], np.uint8).reshape(28, 28), cmap="gray", vmin=0, vmax=255)
        axes[i, j].axis("off")
    axes[i, 0].set_title(name, fontsize=8, loc="left")
plt.tight_layout()
plt.show()

Mỗi hàng là 8 mẫu của một bộ khử nhiễu, cùng 8 nhiễu ban đầu. Hàng đầu (lý tưởng) toàn là ảnh huấn luyện chép nguyên.
Mảng nhỏ ($P = 3$) cho những mảnh vụn rời rạc: mỗi điểm ảnh tự chọn một ảnh khác, không có gì giữ các nét liền nhau.
Mảng vừa ($P = 5$ đến 9) cho những nét chữ mới, liền mạch, đôi khi lai giữa hai chữ số. Mảng lớn ($P = 13$) lại gần
ảnh huấn luyện hơn (6% là bản sao): mảng càng lớn thì bộ khử nhiễu càng gần bản lý tưởng. Hàng cuối thêm bất biến
tịnh tiến (mảng lấy từ **bất kỳ vị trí nào** của bất kỳ ảnh nào, trên 256 ảnh): với $P = 5$ cố định, kết quả là những
nét rời rạc, không thành chữ.

Còn phản biện của Lukoianov và cộng sự: bộ khử nhiễu **tuyến tính** tối ưu (chỉ dùng trung bình và hiệp phương sai của
dữ liệu, không có mạng nào) đã cục bộ sẵn chưa? Ô dưới vẽ các trọng số của nó cho điểm ảnh giữa ảnh, ở $t = 0{,}5$.

In [ ]:
lin = LOCAL["linear"]
row = np.array(lin["row"]).reshape(28, 28)
fig, ax = plt.subplots(figsize=(3.6, 3.2))
lim = np.abs(row).max()
im = ax.imshow(row, cmap=DIVERGING, vmin=-lim, vmax=lim)
ax.plot(*lin["pixel"][::-1], marker="+", color="black", ms=10)
ax.set_title(f"trọng số cho điểm ảnh {tuple(lin['pixel'])}, t = {vn(lin['t'], 1)}", fontsize=9)
ax.axis("off")
fig.colorbar(im, ax=ax, fraction=0.046)
vn_axes(fig)
plt.show()
for rad, share in zip(lin["radii"], lin["massWithin"]):
    print(f"trong bán kính {rad} điểm ảnh: {pct(share, 0)} tổng độ lớn các trọng số")

Ảnh nhiệt là các trọng số của điểm ảnh đánh dấu "+" (xanh: dương, cam: âm). Chúng dồn quanh chính điểm ảnh đó: gần
một nửa tổng độ lớn nằm trong bán kính 3 điểm ảnh, dù bộ khử nhiễu được phép nhìn cả ảnh. Vậy dữ liệu tự nó đã đẩy
lời giải tối ưu về phía cục bộ, đúng như Lukoianov và cộng sự nói; điều đó không bác bỏ Kamb & Ganguli (cục bộ vẫn là
thứ biến sao chép thành sáng tạo), nó chỉ cho rằng cục bộ không nhất thiết đến từ kiến trúc mạng.

**Kết luận: khớp phần chính, tùy điều kiện phần còn lại.** Bộ khử nhiễu chính xác sao chép 100%; giới hạn nó vào mảng
cục bộ thì hết sao chép (0% với $P \le 9$) và tạo ra nét chữ mới với $P$ khoảng 5–9. Nhưng chất lượng phụ thuộc mạnh
vào cỡ mảng, và ta dùng **một** cỡ mảng cho mọi $t$; Kamb & Ganguli chọn cỡ mảng thay đổi theo $t$, khớp với mạng đã
huấn luyện. Ta không kiểm phần định lượng chính của họ: các máy công thức dự đoán đầu ra của U-Net.

## 4 · Khi nào mô hình khuếch tán bắt đầu sao chép?

**Nhận định** (Bonnaire, Urfin, Biroli & Mézard, NeurIPS 2025): trong lúc huấn luyện có **hai mốc thời gian**. Mô hình
bắt đầu sinh mẫu tốt và *mới* từ một thời điểm $\tau_{\text{gen}}$ gần như không phụ thuộc số ảnh huấn luyện $n$. Nó
chỉ bắt đầu chép lại ảnh huấn luyện muộn hơn, ở $\tau_{\text{mem}}$, và **$\tau_{\text{mem}}$ tăng tỉ lệ với $n$**.
Giữa hai mốc là một cửa sổ mà mô hình tổng quát hóa, và cửa sổ đó rộng ra khi $n$ tăng.

Thí nghiệm, khuếch tán tiềm ẩn của Bài 21 thu nhỏ: dữ liệu là mã 16 chiều của các ảnh MNIST qua bộ mã hóa tự động của
Bài 19; mô hình là một MLP dư (residual), rộng 256, học flow matching; $n$ từ 32 đến 1.024 mã, 3 seed, 40.000 bước.
Phần này chỉ đọc kết quả (mỗi lần chạy mất vài phút, và cần bộ mã hóa của Bài 19). Ở mỗi điểm đo, mô hình sinh 1.000
mẫu, và ta đo:
- **tỉ lệ bản sao:** số mẫu gần mã huấn luyện gần nhất hơn 99% số mã kiểm tra thật (quy tắc của Bài 21);
- **FD:** khoảng cách Fréchet giữa phân phối Gauss khớp với các mẫu và với 1.000 mã kiểm tra (càng nhỏ càng giống; ô
  dưới in khoảng cách giữa hai tập mã thật, làm mức sàn).

### 🤔 Dự đoán trước
Với 32 mã huấn luyện, mô hình bắt đầu chép (10% mẫu là bản sao) sau 500 bước. Với 1.024 mã (nhiều gấp 32 lần), bạn
đoán là sau bao nhiêu bước?

In [ ]:
def steps_text(es):
    """[500, 500, None] → "500, 500, không"."""
    return ", ".join("không" if e is None else vn(e, 0) for e in es)


checks = np.array(MEMO["check"])
print(f"FD giữa hai tập 1.000 mã thật (mức sàn): {vn(MEMO['fdRealVsReal'], 2)}")
onset = {}
for n in MEMO["sizes"]:
    rs = MEMO["runs"][str(n)]
    onset[n] = [next((int(st) for st, c in zip(checks, r_["copy"]) if c >= 0.1), None) for r_ in rs]
    fd = np.median([r_["fd"] for r_ in rs], 0)
    final = [r_["copy"][-1] for r_ in rs]
    print(f"n = {n:5}: chép ≥ 10% từ bước {steps_text(onset[n])} · {pct(min(final), 0)}–{pct(max(final), 0)} bản sao ở bước "
          f"{vn(checks[-1], 0)} · FD ở bước {vn(1000, 0)} / {vn(checks[-1], 0)}: {vn(fd[1], 2)} / {vn(fd[-1], 2)}")  # fmt: skip

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
cmap = plt.get_cmap("viridis")
for k, n in enumerate(MEMO["sizes"]):
    rs = MEMO["runs"][str(n)]
    col = cmap(k / (len(MEMO["sizes"]) - 1))
    ls = "-" if k % 2 == 0 else "--"
    axes[0].plot(checks, np.median([r_["copy"] for r_ in rs], 0), color=col, ls=ls, lw=1.8, label=f"n = {n}")
    axes[1].plot(checks, np.median([r_["fd"] for r_ in rs], 0), color=col, ls=ls, lw=1.8, label=f"n = {n}")
ns = np.array(MEMO["sizes"])
med_onset = np.array([np.median(onset[n]) for n in ns])
axes[2].plot(ns, med_onset, "o", color=BLUE, label="bắt đầu chép (trung vị 3 seed)")
axes[2].plot(ns, med_onset[0] * ns / ns[0], color=GREY, ls=":", label="tỉ lệ thuận với n")
axes[0].set(xscale="log", xlabel="bước", ylabel="tỉ lệ bản sao (trung vị)", title="Sao chép")
axes[1].set(xscale="log", xlabel="bước", ylabel="FD với mã kiểm tra (trung vị)", title="Chất lượng mẫu")
axes[2].set(xscale="log", yscale="log", xlabel="số mã huấn luyện n", ylabel="bước bắt đầu chép", title="τ_mem theo n")
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
axes[0].legend(frameon=False, fontsize=7)
axes[2].legend(frameon=False, fontsize=7)
vn_axes(fig)
plt.tight_layout()
plt.show()

Bên trái: tỉ lệ bản sao theo số bước (thang log), mỗi $n$ một màu, từ tím ($n = 32$) đến vàng ($n = 1.024$), nét liền
và nét đứt xen kẽ. Đường của $n$ càng lớn càng dịch sang phải: mô hình chép càng muộn. Ở giữa: FD theo số bước. Với
$n \ge 128$, FD đã gần mức tốt nhất của nó từ khoảng bước 1.000, rồi chỉ thay đổi chút ít (hơi tăng khi mô hình bắt
đầu chép), trong khi tỉ lệ bản sao vẫn tiếp tục tăng: chất lượng đến sớm, sao chép đến muộn. Bên phải: bước bắt đầu chép theo
$n$ (cả hai trục thang log), cùng đường tỉ lệ thuận (chấm xám). Các điểm bám gần đường đó: mỗi lần $n$ gấp đôi, mô hình
chép muộn đi khoảng gấp đôi (lần cuối nhiều hơn), từ 500 bước ($n = 32$) đến 30.000–40.000 bước ($n = 1.024$).

**Kết luận: khớp.** Bắt đầu sao chép muộn dần tỉ lệ với số dữ liệu huấn luyện, còn chất lượng mẫu đạt sớm và gần như
không đổi. Giới hạn: các điểm đo thưa (500, 1.000, 2.000, …), nên mốc bắt đầu chỉ chính xác đến điểm đo; FD đo trong
không gian mã 16 chiều với 1.000 mẫu, nên chỉ so sánh được trong thí nghiệm này. Với $n = 1.024$, mô hình chưa chép
xong sau 40.000 bước (12–14%, vẫn đang tăng): với đủ dữ liệu so với sức chứa của mô hình, $\tau_{\text{mem}}$ vượt qua
thời gian huấn luyện thực tế, và mô hình không bao giờ kịp chép, như bài báo dự đoán.

## 5 · Sụp đổ mô hình: thay dữ liệu hay giữ lại?

**Nhận định.** Shumailov và cộng sự (2024, *Nature*): nếu mỗi thế hệ mô hình mới chỉ học từ dữ liệu do thế hệ trước
sinh ra, phần đuôi của phân phối dần biến mất và mô hình thoái hóa: **sụp đổ mô hình**. Gerstgrasser và cộng sự (2024)
đáp lại: điều đó xảy ra khi dữ liệu tổng hợp **thay thế** dữ liệu thật; nếu mỗi thế hệ **thêm** dữ liệu tổng hợp vào
kho dữ liệu đã có (kể cả dữ liệu thật), sai số được chặn lại.

Thí nghiệm đơn giản nhất: một phân phối chuẩn 1 chiều. Thế hệ 0 ước lượng trung bình và độ lệch chuẩn từ 100 mẫu thật
của $\mathcal{N}(0, 1)$. Mỗi thế hệ sau rút 100 mẫu từ ước lượng hiện tại rồi ước lượng lại, theo hai cách:
- **thay thế:** chỉ dùng 100 mẫu mới;
- **tích lũy:** dùng tất cả các mẫu từ trước đến nay (100 mẫu thật cộng 100 mẫu của mỗi thế hệ).

### 🤔 Dự đoán trước
Mỗi lần ước lượng lại chỉ có sai số ngẫu nhiên nhỏ, không có sai lệch hệ thống nào. Sau 200 thế hệ thay thế, độ lệch
chuẩn ước lượng sẽ quanh 1, nhỏ hơn hẳn 1, hay lớn hơn hẳn 1?

In [ ]:
def gauss_generations(mode, rng, generations=200, n=100, seeds=200):
    sig = np.empty((seeds, generations + 1))
    for s in range(seeds):
        pool = rng.standard_normal(n)
        mu, sd = pool.mean(), pool.std()
        sig[s, 0] = sd
        for g in range(1, generations + 1):
            new = rng.normal(mu, sd, n)
            pool = new if mode == "replace" else np.concatenate([pool, new])
            mu, sd = pool.mean(), pool.std()
            sig[s, g] = sd
    return sig


rng = np.random.default_rng(0)  # một dòng số ngẫu nhiên cho cả hai cách, theo thứ tự này (như script)
gauss = {m: gauss_generations(m, rng) for m in ("replace", "accumulate")}
for m, label in (("replace", "thay thế"), ("accumulate", "tích lũy")):
    med = np.median(gauss[m], 0)
    print(f"{label:9}: σ trung vị ở thế hệ 0 / 10 / 50 / 200 = " + " / ".join(vn(med[g], 3) for g in (0, 10, 50, 200)) +
          f" · tỉ lệ seed có σ < 0,1 ở thế hệ 200: {pct((gauss[m][:, 200] < 0.1).mean(), 0)}")  # fmt: skip
    stored = COLLAPSE["gauss"][m]["sigma"][1]
    assert all(abs(med[g] - stored[g]) < 1e-3 for g in (0, 10, 50, 200)), "khác với kết quả đã lưu"

Thay thế: độ lệch chuẩn co dần, 0,62 ở thế hệ 50 và 0,14 ở thế hệ 200; ở 40% số seed, nó dưới 0,1. Không có sai lệch
hệ thống ở từng bước, nhưng **sai số ngẫu nhiên tích lũy lại**: độ lệch chuẩn mẫu hơi thấp hơn σ thật một chút mỗi
lần, và ở thế hệ sau không còn dữ liệu thật nào kéo nó về. Tích lũy: độ lệch chuẩn đứng yên ở 0,98, vì 100 mẫu thật
luôn có mặt và các mẫu mới chỉ là phần thêm vào.

Thí nghiệm thứ hai, gần với ngôn ngữ hơn: mô hình bigram ký tự của *Truyện Kiều* (xác suất của ký tự tiếp theo khi
biết ký tự hiện tại, có làm trơn để không ký tự nào có xác suất 0). Mỗi thế hệ sinh ra khoảng 96 nghìn ký tự, bằng độ
dài văn bản thật, rồi đếm lại. Ta đo số bit mỗi ký tự trên 254 câu thơ cuối (không dùng để huấn luyện), và tỉ lệ các
cặp ký tự của văn bản kiểm tra mà mô hình còn "biết" (đã đếm được ít nhất một lần).

In [ ]:
body = KIEU_RAW.read_bytes().decode("utf-8").split("<poem>")[1].split("</poem>")[0]
lines = [re.sub(r"\{\{số\|\d+\}\}", "", ln).strip() for ln in body.split("\n")]
text = "\n".join(ln for ln in lines if ln) + "\n"
assert hashlib.md5(text.encode()).hexdigest() == "dd94ee390bfc543287b11af62b2f2fc8"
CHARS = sorted(set(text))
V = len(CHARS)
stoi = {ch: i for i, ch in enumerate(CHARS)}
ids = np.array([stoi[ch] for ch in text])
cut = len("\n".join(text[:-1].split("\n")[:3000])) + 1
kieu_tr, kieu_te = ids[:cut], ids[cut:]
test_pairs = list(set(zip(kieu_te[:-1].tolist(), kieu_te[1:].tolist())))
ALPHA = 0.01


def bigram_counts(seq):
    C = np.zeros((V, V))
    np.add.at(C, (seq[:-1], seq[1:]), 1)
    return C


def bigram_probs(C):
    P = C + ALPHA
    return P / P.sum(1, keepdims=True)


def bigram_sample(P, n, rng, start):
    cum = P.cumsum(1)
    out = np.empty(n, np.int64)
    c, u = start, rng.random(n)
    for i in range(n):
        c = min(int(np.searchsorted(cum[c], u[i])), V - 1)
        out[i] = c
    return out


def bigram_stats(C):
    bits = -np.log2(bigram_probs(C)[kieu_te[:-1], kieu_te[1:]]).mean()
    seen = np.mean([C[a, b] > 0 for a, b in test_pairs])
    return bits, seen


def bigram_generations(mode, generations=4, seed=0):
    rng = np.random.default_rng(100 + seed)
    C = bigram_counts(kieu_tr)
    out = [bigram_stats(C)]
    for _ in range(generations):
        new = bigram_sample(bigram_probs(C), len(kieu_tr), rng, start=int(kieu_tr[0]))
        C = bigram_counts(new) if mode == "replace" else C + bigram_counts(new)
        out.append(bigram_stats(C))
    return out


print(f"{V} ký tự khác nhau; {vn(len(kieu_tr), 0)} ký tự huấn luyện, {vn(len(kieu_te), 0)} ký tự kiểm tra")
for m, label in (("replace", "thay thế"), ("accumulate", "tích lũy")):
    st = bigram_generations(m)
    print(f"{label:9}: bit/ký tự thế hệ 0 → 4: {vn(st[0][0], 3)} → {vn(st[-1][0], 3)} · "
          f"cặp ký tự kiểm tra còn biết: {pct(st[0][1])} → {pct(st[-1][1])}")  # fmt: skip
    assert abs(st[-1][0] - COLLAPSE["bigram"][m]["bits"][0][4]) < 1e-3, "khác với kết quả đã lưu"

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
gens = np.arange(COLLAPSE["gauss"]["generations"] + 1)
for m, col, ls, label in (("replace", ORANGE, "--", "thay thế"), ("accumulate", BLUE, "-", "tích lũy")):
    q25, q50, q75 = (np.array(v) for v in COLLAPSE["gauss"][m]["sigma"])
    axes[0].plot(gens, q50, color=col, ls=ls, lw=1.8, label=label)
    axes[0].fill_between(gens, q25, q75, color=col, alpha=0.15)
    bits = np.array(COLLAPSE["bigram"][m]["bits"])
    g2 = np.arange(bits.shape[1])
    axes[1].plot(g2, np.median(bits, 0), color=col, ls=ls, lw=1.8, marker="o" if m == "replace" else "s", ms=4,
                 label=label)  # fmt: skip
for m, label in (
    ("replace", "thay thế"),
    ("accumulate", "tích lũy"),
    ("accumulate-fixed", "tích lũy, cắt về cỡ ban đầu"),
):
    b = np.median(np.array(COLLAPSE["bigram"][m]["bits"]), 0)
    seen = np.median(np.array(COLLAPSE["bigram"][m]["testSeen"]), 0)
    print(f"{label:28}: thế hệ 15 (trung vị 5 seed): {vn(b[-1], 3)} bit/ký tự, cặp kiểm tra còn biết {pct(seen[-1])}")
bits_fixed = np.array(COLLAPSE["bigram"]["accumulate-fixed"]["bits"])
axes[1].plot(np.arange(bits_fixed.shape[1]), np.median(bits_fixed, 0), color=GREEN, ls="-.", lw=1.8, marker="^", ms=4,
             label="tích lũy, cắt về cỡ ban đầu")  # fmt: skip
axes[0].set(xlabel="thế hệ", ylabel="σ ước lượng (trung vị, 25–75%)", title="Phân phối chuẩn, n = 100")
axes[1].set(xlabel="thế hệ", ylabel="bit/ký tự trên câu thơ kiểm tra", title="Bigram của Truyện Kiều")
axes[0].legend(frameon=False)
axes[1].legend(frameon=False, fontsize=8)
vn_axes(fig)
plt.tight_layout()
plt.show()

Bên trái: độ lệch chuẩn ước lượng qua 200 thế hệ (đường là trung vị của 200 seed, dải mờ là khoảng 25–75%). Thay thế
(cam, nét đứt) trôi dần về 0 và dải mở rộng; tích lũy (xanh, nét liền) nằm yên. Bên phải: số bit mỗi ký tự trên văn
bản kiểm tra qua 15 thế hệ, 5 seed (các con số in ở trên). Thay thế tệ dần đều đặn, và tỉ lệ cặp ký tự kiểm tra mà mô
hình còn biết giảm dần: các cặp hiếm biến mất trước. Tích lũy gần như đứng yên. Đường xanh lá là phép thử của
Gerstgrasser: vẫn giữ mọi thế hệ, nhưng cắt kho dữ liệu về đúng cỡ ban đầu, để tách "nhiều dữ liệu hơn" khỏi "giữ dữ liệu
cũ". Nó tệ hơn tích lũy một chút nhưng vẫn ổn định: thứ chặn sụp đổ là việc **dữ liệu thật không bị bỏ đi**, không phải
chỉ là có nhiều dữ liệu hơn.

**Kết luận: khớp cả hai.** Thay thế dữ liệu thì sụp đổ (Shumailov), giữ dữ liệu cũ thì không (Gerstgrasser): hai bài
báo không mâu thuẫn, chúng mô tả hai cách làm khác nhau. Một chi tiết của thí nghiệm này: không ký tự nào biến mất hẳn,
vì phép làm trơn luôn để lại một xác suất nhỏ; cái mất đi là các **cặp** ký tự hiếm.

## 6 · RL's Razor: vì sao học tăng cường quên ít hơn SFT

**Nhận định** (Shenfeld, Pari & Agrawal, 2025): khi huấn luyện tiếp một mô hình cho một việc mới, học tăng cường (RL) và
SFT có thể đạt cùng độ chính xác trên việc mới, nhưng RL quên **ít hơn** những gì mô hình đã biết. Họ cho rằng mức quên
được dự đoán bởi một con số duy nhất: độ phân kỳ $\mathrm{KL}(\pi_0 \,\|\, \pi)$ giữa mô hình gốc và mô hình mới, đo
trên đầu vào của việc mới. Và SFT trên cách gán nhãn gần $\pi_0$ nhất ("SFT tối ưu") còn quên ít hơn cả RL. Đây là một
bài báo, chưa được ai kiểm lại độc lập.

Ta dựng lại đúng thí nghiệm nhỏ của bài báo, cỡ nhỏ hơn. Một MLP 785 → 256 → 128 → 10, với một **cờ** ở cuối đầu vào
(+1: việc chẵn lẻ, −1: Fashion-MNIST):
- **tiền huấn luyện** chung trên 500 ảnh chữ số gán nhãn kiểu "ParityMNIST" (mỗi ảnh một nhãn ngẫu nhiên **cùng tính
  chẵn lẻ** với chữ số, cố định) và 500 ảnh Fashion-MNIST với nhãn thật;
- **việc mới**: lại là chẵn lẻ, nhưng giờ mọi nhãn cùng tính chẵn lẻ đều đúng, nên mỗi ảnh có **5 đáp án đúng**. Chính
  điều này làm RL và SFT khác nhau: SFT phải chọn một cách gán nhãn, RL được tự chọn.

Ba cách huấn luyện tiếp, đều trên 500 ảnh chữ số mới:
- **SFT:** chẵn → nhãn 0, lẻ → nhãn 1;
- **RL:** lấy 8 mẫu nhãn cho mỗi ảnh, thưởng 1 nếu đúng tính chẵn lẻ, REINFORCE với đường cơ sở của nhóm (kiểu GRPO,
  Bài 26);
- **SFT tối ưu:** nhãn mềm $q = \pi_0$ chỉ giữ lại 5 nhãn đúng rồi chuẩn hóa lại, tức phân phối đúng gần $\pi_0$ nhất.

In [ ]:
def flagged(x, v):
    x = torch.from_numpy(x.reshape(len(x), -1).astype(np.float32) / 255)
    return torch.cat([x, torch.full((len(x), 1), float(v))], 1)


PARITY = torch.tensor([[1.0 if (j % 2) == (d % 2) else 0.0 for j in range(10)] for d in range(10)])
Tpar, ypar = flagged(mnist_te[:2000], 1), torch.from_numpy(mnist_yte[:2000])
Tfash, yfash = flagged(fash_te[:2000], -1), torch.from_numpy(fash_yte[:2000])
Xm_all, ym_all = torch.from_numpy(mnist_tr.reshape(-1, 784).astype(np.float32) / 255), torch.from_numpy(mnist_ytr)
Xf_all, yf_all = torch.from_numpy(fash_tr.reshape(-1, 784).astype(np.float32) / 255), torch.from_numpy(fash_ytr)


def num_lr(v):
    return f"{vn(v * 10 ** -math.floor(math.log10(v)), 0)}·10{str(math.floor(math.log10(v))).translate(SUP)}"


def razor_net(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(785, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))


def flag(x, v):
    return torch.cat([x, torch.full((len(x), 1), float(v))], 1)


def pretrain(seed=0, steps=500, n=500):
    """Giống hệt reel/research/razor.py: cùng thứ tự rút số ngẫu nhiên."""
    g = torch.Generator().manual_seed(seed)
    im = torch.randperm(len(Xm_all), generator=g)[:n]
    jf = torch.randperm(len(Xf_all), generator=g)[:n]
    lab_par = (ym_all[im] % 2) + 2 * torch.randint(0, 5, (n,), generator=g)
    X = torch.cat([flag(Xm_all[im], 1), flag(Xf_all[jf], -1)])
    Y = torch.cat([lab_par, yf_all[jf]])
    m = razor_net(seed)
    opt = torch.optim.Adam(m.parameters(), 1e-3)
    for _ in range(steps):
        b = torch.randint(0, len(X), (128,), generator=g)
        loss = nn.functional.cross_entropy(m(X[b]), Y[b])
        opt.zero_grad()
        loss.backward()
        opt.step()
    return m, set(im.tolist())


def measure(m, p0_par):
    with torch.no_grad():
        lp = torch.log_softmax(m(Tpar), 1)
        new = (lp.exp() * PARITY[ypar]).sum(1).mean().item()  # xác suất dồn vào các nhãn đúng tính chẵn lẻ
        kl = (p0_par * (torch.log(p0_par + 1e-30) - lp)).sum(1).mean().item()
        old = (m(Tfash).argmax(1) == yfash).float().mean().item()
    return new, old, max(kl, 0.0)


def finetune(method, base_state, used, seed=0, steps=200, lr=1e-3, n=500):
    m = razor_net(seed)
    m.load_state_dict(base_state)
    g = torch.Generator().manual_seed(100 + seed)
    pool = torch.tensor([i for i in torch.randperm(len(Xm_all), generator=g).tolist() if i not in used][:n])
    X, digit = flag(Xm_all[pool], 1), ym_all[pool]
    with torch.no_grad():
        p0_par, p0_train = torch.softmax(m(Tpar), 1), torch.softmax(m(X), 1)
    if method == "sft":
        target = nn.functional.one_hot(digit % 2, 10).float()
    elif method == "oracle":
        q = p0_train * PARITY[digit]
        target = q / q.sum(1, keepdim=True)
    before = measure(m, p0_par)
    opt = torch.optim.Adam(m.parameters(), lr)
    for _ in range(steps):
        b = torch.randint(0, len(X), (64,), generator=g)
        logits = m(X[b])
        if method == "rl":
            lp = torch.log_softmax(logits, 1)
            with torch.no_grad():
                a = torch.multinomial(lp.exp(), 8, replacement=True, generator=g)
                rew = PARITY[digit[b]].gather(1, a)
                adv = rew - rew.mean(1, keepdim=True)
            loss = -(adv * lp.gather(1, a)).mean()
        else:
            loss = -(target[b] * torch.log_softmax(logits, 1)).sum(1).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    return before, measure(m, p0_par)


base_model, used = pretrain()
base_state = {k: v.clone() for k, v in base_model.state_dict().items()}
print("mô hình gốc đã tiền huấn luyện (500 bước)")

### 🤔 Dự đoán trước
Sau 200 bước huấn luyện tiếp, ba cách đều đạt khoảng cùng độ chính xác trên việc chẵn lẻ. Cách nào giữ được Fashion-MNIST
tốt nhất, và cách nào quên nhiều nhất?

In [ ]:
names = {"sft": "SFT (chẵn → 0, lẻ → 1)", "rl": "RL (kiểu GRPO)", "oracle": "SFT tối ưu"}
for mth in ("sft", "rl", "oracle"):
    (n0, o0, _), (n1, o1, kl) = finetune(mth, base_state, used)
    print(f"{names[mth]:23} chẵn lẻ {pct(n0)} → {pct(n1)} · Fashion-MNIST {pct(o0)} → {pct(o1)} · KL(π₀‖π) {vn(kl, 2)}")
    stored = RAZOR["live"]["runs"][mth]
    assert abs(o1 - stored["old"][-1]) < 0.02 and abs(kl - stored["kl"][-1]) < 0.05 * max(1, stored["kl"][-1])

SFT quên gần hết Fashion-MNIST, RL quên một phần, SFT tối ưu gần như không quên. Và KL đi theo đúng thứ tự đó. Bản đầy
đủ: 3 cách × 3 tốc độ học × 3 seed, đo trong suốt quá trình huấn luyện (số bước in ở dưới). Để so sánh công bằng, ta lấy
các điểm đo mà việc mới đã đạt 88–90%:

In [ ]:
pts = {m: [] for m in ("sft", "rl", "oracle")}
for x in RAZOR["runs"]:
    base_old = x["old"][0]
    pts[x["method"]] += [(k, base_old - o, n) for n, o, k in zip(x["new"][1:], x["old"][1:], x["kl"][1:])]
for m, mp in pts.items():
    sel = [p for p in mp if 0.88 <= p[2] <= 0.90]
    f = np.array([p[1] for p in sel])
    print(f"{names[m]:23} {len(sel):3} điểm đo · mức quên trung vị {pct(np.median(f))} (từ {pct(f.min())} đến {pct(f.max())}) "
          f"· KL trung vị {vn(np.median([p[0] for p in sel]), 2)}")  # fmt: skip
print(f"({RAZOR['steps']} bước mỗi lần chạy, đo mỗi {RAZOR['every']} bước; việc mới đạt tối đa "
      f"{pct(max(max(x['new']) for x in RAZOR['runs']))})")  # fmt: skip
print(f"\nmức quên theo KL, khớp bậc hai trên {sum(map(len, pts.values()))} điểm đo: R² = {vn(RAZOR['fit']['r2'], 3)}")


def r2_quad(x, y):
    A = np.stack([np.ones_like(x), x, x * x], 1)
    c, *_ = np.linalg.lstsq(A, y, rcond=None)
    e = y - A @ c
    return 1 - (e @ e) / ((y - y.mean()) @ (y - y.mean()))


c0, c1, c2 = RAZOR["fit"]["coef"]
for x_lr in sorted({x["lr"] for x in RAZOR["runs"]}):
    res = [
        (x["old"][0] - o) - (c0 + c1 * k + c2 * k * k)
        for x in RAZOR["runs"]
        if x["method"] == "rl" and x["lr"] == x_lr
        for k, o in zip(x["kl"][1:], x["old"][1:])
    ]
    print(f"RL, η {num_lr(x_lr)}: trung bình nằm {'trên' if np.mean(res) > 0 else 'dưới'} đường chung "
          f"{vn(100 * abs(np.mean(res)), 1)} điểm")  # fmt: skip
k_old = np.array([k for x in RAZOR["runs"] for k in x["klOld"][1:]])
forget = np.array([x["old"][0] - o for x in RAZOR["runs"] for o in x["old"][1:]])
print(f"cùng phép khớp, nhưng với KL đo trên ảnh Fashion-MNIST: R² = {vn(r2_quad(k_old, forget), 3)}")

fig, ax = plt.subplots(figsize=(7, 4))
for m, col, mk in (("sft", ORANGE, "s"), ("rl", BLUE, "o"), ("oracle", GREEN, "^")):
    k = np.array([p[0] for p in pts[m]])
    f = np.array([p[1] for p in pts[m]])
    ax.scatter(np.maximum(k, 1e-3), f, s=10, color=col, marker=mk, alpha=0.5, label=names[m])
xs = np.logspace(-3, math.log10(60), 200)
c0, c1, c2 = RAZOR["fit"]["coef"]
ax.plot(xs, c0 + c1 * xs + c2 * xs**2, color="black", lw=1, label="khớp bậc hai")
ax.set(xscale="log", xlabel="KL(π₀ ‖ π) trên ảnh chẵn lẻ", ylabel="mức quên Fashion-MNIST (điểm phần trăm)")
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: vn(100 * v, 0)))
ax.legend(frameon=False, fontsize=8)
vn_axes(fig)
plt.show()

Mỗi điểm là một lần đo (một cách huấn luyện, một tốc độ học, một seed, một bước): trục ngang là KL so với mô hình gốc
(thang log), trục dọc là mức tụt trên Fashion-MNIST. SFT (cam, vuông) nằm ở bên phải và trên cao, SFT tối ưu (xanh lá,
tam giác) ở bên trái và dưới thấp, RL (xanh, tròn) ở giữa — và cả ba nằm **gần** một đường cong chung. Mức quên phụ
thuộc chủ yếu vào việc mô hình đã đi xa khỏi $\pi_0$ bao nhiêu, hơn là vào "đó là RL hay SFT". RL quên ít hơn vì nó lấy
mẫu từ chính nó: nó chỉ được thưởng cho các đáp án nó đã có xu hướng chọn, nên nó đi về phía đáp án đúng gần $\pi_0$ mà
không cần ai chỉ.

Hai chỗ đường cong không khớp hoàn toàn (in ở ô trên): RL với tốc độ học lớn nhất nằm **trên** đường chung khoảng 10
điểm, tức nó quên nhiều hơn mức KL của nó dự đoán; và KL đo trên chính ảnh Fashion-MNIST dự đoán mức quên gần tốt
bằng. Nên "một con số KL duy nhất" ở đây là một tóm tắt tốt, chưa phải một định luật.

**Kết luận: khớp, với một điều kiện.** Thứ tự quên SFT > RL > SFT tối ưu đúng ở mọi tốc độ học, và KL so với mô hình
gốc dự đoán phần lớn mức quên ($R^2 = 0{,}93$; bài báo: 0,96). Nhưng ở tốc độ học lớn, RL lệch khỏi đường chung.
Giới hạn: một bài toán đồ chơi, một kiến trúc; và việc mới không vượt quá mức in ở trên, vì mô hình nhỏ này chỉ nhận ra
chữ số đến mức đó.

## 7 · Chuỗi suy nghĩ trên bài toán chẵn lẻ

**Nhận định** (Kim & Suzuki, ICLR 2025): học thẳng **tính chẵn lẻ thưa** — XOR của $k$ bit ở những vị trí bí mật trong
$n$ bit — là khó với phương pháp gradient: mất mát đứng yên ở mức đoán mò rất lâu. Nếu mô hình được dạy viết ra các kết
quả trung gian (tính chẵn lẻ của 1, 2, …, $k$ bit bí mật đầu tiên, mỗi kết quả một token), mỗi bước chỉ còn là XOR của
2 bit, và một transformer nhỏ học được rất nhanh. Họ chứng minh điều này cho transformer một tầng.

Thí nghiệm: $n = 20$ bit, $k = 4$ vị trí bí mật cố định, đầu vào ngẫu nhiên mới ở mỗi bước (nên không có tập huấn
luyện nào để thuộc lòng). Transformer nhân quả 2 khối, $d = 64$, 4 đầu.

In [ ]:
SECRET = torch.tensor(COT["tasks"]["n20k4"]["secret"])
N_BITS = COT["tasks"]["n20k4"]["n"]


class Tiny(nn.Module):
    def __init__(self, T, d=64):
        super().__init__()
        self.emb = nn.Embedding(2, d)
        self.pos = nn.Parameter(torch.randn(T, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 4, 2 * d, dropout=0.0, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.head = nn.Linear(d, 2)
        self.register_buffer("mask", torch.triu(torch.ones(T, T, dtype=torch.bool), 1))

    def forward(self, x):
        L = x.shape[1]
        return self.head(self.enc(self.emb(x) + self.pos[:L], mask=self.mask[:L, :L], is_causal=True))


def parity_run(cot, steps, seed=0, every=100):
    """Giống hệt reel/research/cot.py."""
    n, S, k = N_BITS, SECRET, len(SECRET)
    torch.manual_seed(seed)
    m = Tiny(n + k)
    opt = torch.optim.AdamW(m.parameters(), 1e-3)
    g, ge = torch.Generator().manual_seed(1000 + seed), torch.Generator().manual_seed(2000 + seed)
    acc = []
    for s in range(1, steps + 1):
        x = torch.randint(0, 2, (256, n), generator=g)
        bits = x[:, S]
        tgt = torch.cumsum(bits, 1) % 2 if cot else bits.sum(1, keepdim=True) % 2
        inp = torch.cat([x, tgt[:, :-1]], 1) if cot else x
        loss = nn.functional.cross_entropy(m(inp)[:, n - 1 :].reshape(-1, 2), tgt.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if s % every == 0:
            with torch.no_grad():
                x = torch.randint(0, 2, (2000, n), generator=ge)
                ans = x[:, S].sum(1) % 2
                if cot:  # sinh lần lượt k token, token cuối là đáp án
                    inp = x
                    for _ in range(k):
                        inp = torch.cat([inp, m(inp)[:, -1].argmax(1, keepdim=True)], 1)
                    pred = inp[:, -1]
                else:
                    pred = m(x)[:, -1].argmax(1)
                acc.append((pred == ans).float().mean().item())
    return acc


print(f"{N_BITS} bit, vị trí bí mật {SECRET.tolist()}")

### 🤔 Dự đoán trước
Học thẳng đáp án, mô hình đúng bao nhiêu phần trăm sau 300 bước? Viết 4 token trung gian rồi mới trả lời thì sao?
(Ô dưới chỉ chạy trực tiếp cách có chuỗi suy nghĩ; cách học thẳng tốn cùng thời gian mà không đi tới đâu, nên ta
đọc kết quả của nó, cùng seed, từ tệp đã lưu.)

In [ ]:
cot_live = parity_run(True, 300, every=300)  # chỉ đo ở bước cuối, cho nhanh
saved = {k: COT["runs"][k][0]["acc"][:3] for k in ("n20k4-direct", "n20k4-cot")}  # seed 0, cả hai cách
print(
    "trả lời thẳng  (đã lưu): độ chính xác sau 100 / 200 / 300 bước = "
    + " / ".join(pct(a) for a in saved["n20k4-direct"])
)
print(
    "chuỗi suy nghĩ (đã lưu): độ chính xác sau 100 / 200 / 300 bước = " + " / ".join(pct(a) for a in saved["n20k4-cot"])
)
print(f"chuỗi suy nghĩ, vừa chạy: sau 300 bước = {pct(cot_live[-1])}")
assert abs(cot_live[-1] - saved["n20k4-cot"][2]) < 0.02, "khác với kết quả đã lưu"

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for key, col, ls, label in (("n20k4-direct", ORANGE, "--", "n 20, k 4 · trả lời thẳng"),
                            ("n20k4-cot", BLUE, "-", "n 20, k 4 · chuỗi suy nghĩ"),
                            ("n24k6-direct", GREY, ":", "n 24, k 6 · trả lời thẳng"),
                            ("n24k6-cot", GREEN, "-.", "n 24, k 6 · chuỗi suy nghĩ")):  # fmt: skip
    for i, r_ in enumerate(COT["runs"][key]):
        steps = np.arange(1, len(r_["acc"]) + 1) * COT["eval"]
        ax.plot(steps, r_["acc"], color=col, ls=ls, lw=1.4, label=label if i == 0 else None)
ax.set(xscale="log", xlabel="bước huấn luyện", ylabel="độ chính xác (2.000 đầu vào mới)", ylim=(0.4, 1.02))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: pct(v, 0)))
ax.legend(frameon=False, fontsize=8)
vn_axes(fig)
plt.show()
for key in COT["runs"]:
    rs = COT["runs"][key]
    print(
        f"{key:13}: đạt 99% ở bước {steps_text([r_['solved'] for r_ in rs])} · cuối cùng {', '.join(pct(r_['acc'][-1]) for r_ in rs)}"
    )

Mỗi đường là một seed (3 seed mỗi cấu hình), trục ngang là số bước (thang log). Học thẳng (cam và xám, nét đứt và nét
chấm) nằm ở 50% suốt 10.000 bước, ở cả 3 seed và cả hai bài toán: mô hình không tìm ra các vị trí bí mật. Có chuỗi suy
nghĩ (xanh và xanh lá), nó giải xong trong 300 bước với $k = 4$, và 500–700 bước với $k = 6$.

**Kết luận: khớp.** Viết ra các bước trung gian biến một bài toán mà gradient gần như không tìm được đường vào thành
một chuỗi bài toán dễ. Giới hạn: ở đây chuỗi suy nghĩ được **dạy sẵn** (teacher forcing trên các bước đúng); bài báo
cũng xét trường hợp này. Mô hình không tự nghĩ ra cách chia nhỏ bài toán.

## Tóm tắt
- **Sao chép hay sáng tạo — khớp phần chính.** Bộ khử nhiễu chính xác chỉ sao chép (100%); giới hạn nó vào mảng cục
  bộ thì không còn bản sao nào với $P \le 9$, và mảng cỡ 5–9 tạo ra nét chữ mới. Chất lượng phụ thuộc mạnh vào cỡ mảng.
- **Khi nào bắt đầu sao chép — khớp.** Mốc bắt đầu chép tăng gần tỉ lệ với số dữ liệu (500 bước với 32 mã, 30.000–
  40.000 với 1.024), còn chất lượng mẫu đạt sớm và gần như không đổi.
- **Sụp đổ mô hình — khớp cả hai phía.** Thay dữ liệu thật bằng dữ liệu tổng hợp thì sụp đổ; giữ dữ liệu cũ lại thì
  không. Hai bài báo mô tả hai cách làm khác nhau, không mâu thuẫn.
- **RL's Razor — khớp.** SFT quên nhiều nhất, RL ít hơn, SFT tối ưu ít nhất, và mức quên của cả ba nằm trên một đường
  cong theo $\mathrm{KL}(\pi_0 \| \pi)$ ($R^2 = 0{,}93$).
- **Chuỗi suy nghĩ — khớp.** Học thẳng tính chẵn lẻ của 4 hay 6 bit bí mật đứng ở mức đoán mò suốt 10.000 bước; viết
  các bước trung gian thì giải xong trong vài trăm bước.

## Câu hỏi ôn tập
1. Vì sao bộ khử nhiễu chính xác chỉ có thể trả về ảnh huấn luyện?
2. Vì sao mảng quá nhỏ cho ra mảnh vụn, còn mảng quá lớn lại gần sao chép?
3. Trong thí nghiệm phân phối chuẩn, mỗi bước ước lượng không có sai lệch hệ thống. Vậy vì sao độ lệch chuẩn vẫn trôi
   về 0 khi thay thế dữ liệu?
4. Vì sao bài toán chẵn lẻ có "5 đáp án đúng" lại là điều kiện để RL và SFT quên khác nhau?
5. RL không được ai chỉ cho đáp án gần $\pi_0$ nhất. Vì sao nó vẫn đi gần đáp án đó hơn SFT?
6. Vì sao học thẳng tính chẵn lẻ thưa lại khó cho gradient, còn từng bước XOR hai bit thì dễ?

<details><summary>Gợi ý đáp án</summary>

1. Nó là trung bình có trọng số của các ảnh huấn luyện, và khi $t \to 0$ các trọng số dồn hết vào ảnh gần nhất.
2. Mảng nhỏ thì mỗi điểm ảnh chọn ảnh huấn luyện riêng mà không có gì giữ các nét liền nhau; mảng lớn thì các điểm ảnh
   kề nhau nhìn gần như cùng một vùng, nên chọn cùng một ảnh, và lời giải tiến về bản lý tưởng.
3. Mỗi thế hệ có sai số ngẫu nhiên, và độ lệch chuẩn mẫu hơi thấp hơn σ thật một chút; các sai số đó cộng dồn như một
   bước ngẫu nhiên, mà không có dữ liệu thật nào kéo về.
4. Khi chỉ có một đáp án đúng, mọi cách huấn luyện đều phải đi tới cùng một phân phối; khi có nhiều đáp án đúng, mỗi
   cách đi tới một phân phối khác, và chúng cách $\pi_0$ khác nhau.
5. Nó học từ các mẫu của chính nó, nên chỉ được thưởng cho những đáp án đúng mà nó vốn đã hay chọn, và đẩy xác suất
   của chúng lên theo tỉ lệ đó.
6. Chừng nào chưa có đủ cả $k$ bit, mọi tổ hợp thiếu một bit đều không tương quan gì với đáp án, nên gradient gần như
   bằng 0; XOR hai bit thì mỗi bit đã đúng vị trí làm mất mát giảm ngay.
</details>

## Tìm hiểu thêm
- M. Kamb & S. Ganguli (2024), [An analytic theory of creativity in convolutional diffusion
  models](https://arxiv.org/abs/2412.20292); A. Lukoianov và cộng sự (2025), [Locality in Image Diffusion Models Emerges
  from Data Statistics](https://arxiv.org/abs/2509.09672).
- T. Bonnaire, R. Urfin, G. Biroli & M. Mézard (2025), [Why Diffusion Models Don't Memorize: The Role of Implicit
  Dynamical Regularization in Training](https://arxiv.org/abs/2505.17638).
- I. Shumailov và cộng sự (2023), [The Curse of Recursion: Training on Generated Data Makes Models
  Forget](https://arxiv.org/abs/2305.17493); M. Gerstgrasser và cộng sự (2024), [Is Model Collapse Inevitable? Breaking
  the Curse of Recursion by Accumulating Real and Synthetic Data](https://arxiv.org/abs/2404.01413).
- I. Shenfeld, J. Pari & P. Agrawal (2025), [RL's Razor: Why Online Reinforcement Learning Forgets
  Less](https://arxiv.org/abs/2509.04259).
- J. Kim & T. Suzuki (2024), [Transformers Provably Solve Parity Efficiently with Chain of
  Thought](https://arxiv.org/abs/2410.08633).

**Quay lại:** [Bài 26 · Từ mô hình ngôn ngữ đến trợ lý](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/hau-huan-luyen/).

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · torch {torch.__version__} · numpy {np.__version__} · "
      f"matplotlib {matplotlib.__version__}")  # fmt: skip